# Thermoelectric residual-learning pipeline — steps 1-6, code and results

This notebook is a single-file record of the pipeline as it stands: the raw
data, the physics baseline, the residual, the independent DFT cross-check,
and the one statistical test run so far on fabrication process. Every code
cell below was actually executed in this session immediately before this
file was assembled (not retyped from memory), and every output cell is the
real, unedited stdout from that run.

**Aim:** test whether the gap between measured thermoelectric behavior and
the idealized single-parabolic-band (SPB) physics prediction (the
"residual") systematically depends on how the material was fabricated,
rather than being unexplained noise.

**Steps covered here:**
1. Load raw Starrydata2 samples + curves
2. Clean the data (the two bug fixes)
3. Fit the SPB physics baseline per (family, carrier-type)
4. Compute the residual = measured − baseline
5. Independent cross-check of the fitted effective mass against DFT (ab initio, no fitting)
6. Test whether the residual is associated with fabrication process (Kruskal-Wallis)


## Steps 1-4: load, clean, fit baseline, compute residual

`build_tiers_cleaned.py` — this is the corrected pipeline (see the two `FIX`
blocks in the docstring). It builds both Tier 1 (all families pooled, the
broad test) and Tier 2 (half-Heusler only, the narrow test with an
independent DFT check available in step 5).

**Fix 1** — the carrier-concentration sanity filter was `1e14 < n < 1e23`
(m⁻³), which excluded most real doped-thermoelectric carrier concentrations
(which sit at 1e23-1e28 m⁻³). Corrected to `1e18 < n < 1e27`.

**Fix 2** — the effective mass m* was originally fit pooling n-type and
p-type carriers together per family, which forced the optimizer to
saturate at its search bounds for most families. Fixed by fitting m*
separately per (family, carrier-sign) group.

In [ ]:
"""
CORRECTED VERSION -- two fixes applied on top of the original build_tiers.py,
found by independently re-deriving the pipeline from raw data in a fresh
session (see README_findings_and_fixes.md for the full writeup):

  FIX 1 (line ~250): the physical sanity filter on carrier concentration was
  `1e14 < n_measured < 1e23` (m^-3). The bulk of genuinely valid doped-TE
  carrier concentrations in this dataset sit at 1e23-1e28 m^-3 (1e17-1e22
  cm^-3) -- so the original filter excluded ~85% of good data and mostly kept
  the atypically-low tail, which is why Tier 1 was only 27 samples / 6
  families and Tier 2 half-Heusler was only 2 samples. Corrected window here
  is 1e18-1e27 m^-3.

  FIX 2 (fit_mstar calls below): the single-parabolic-band fit was pooling
  n-type and p-type carriers under one shared effective mass per family. Since
  conduction- and valence-band effective mass genuinely differ, this pooling
  was forcing the optimizer to saturate at its search bounds (0.02 or 25 m_e)
  for most families instead of finding an interior fit. Fixed by fitting m*
  separately per (family, carrier-sign) group, falling back to the pooled fit
  only when a subgroup has <3 points.

Original steps 2-4:
  2. Assemble & clean Tier 1 (pooled multi-family) and Tier 2 (half-Heusler) datasets
     from the Starrydata2 ThermoelectricMaterials dump.
  3. Compute the SPB/Pisarenko (deformation-potential-scattering) baseline and the
     residual (measured S - baseline S) for every matched (T, n, S) point.
  4. Set the validation protocol:
       - Tier 1: family-wise split (leave-one-family-out folds)
       - Tier 2: leave-one-out CV, sample-level, ready for bootstrap on top (per the
         user's decision: LOO + bootstrap for the small-sample mitigation)

Real data in, real numbers out - nothing here is illustrative/fabricated.
"""
import ast
import json
import warnings

import numpy as np
import pandas as pd
from scipy.integrate import quad
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar

warnings.filterwarnings("ignore")

import os
_HERE = os.path.dirname(os.path.abspath(__file__))
DATA_DIR = _HERE   # expects raw_starrydata2_curves.csv.gz / raw_starrydata2_samples.csv.gz here
UPLOAD_DIR = _HERE  # expects tier2_input_halfheusler_samples.csv here
OUT_DIR = _HERE

KB = 1.380649e-23      # J/K
E = 1.602176634e-19    # C
H = 6.62607015e-34     # J*s
ME = 9.1093837015e-31  # kg

# ---------------------------------------------------------------------------
# 1. Precompute the SPB (deformation-potential-scattering, r=-1/2) transport
#    functions on an eta grid, once, then interpolate for speed.
#    n(eta) reduced  = F_{1/2}(eta)
#    S(eta)          = (kB/e) [ 2 F1(eta)/F0(eta) - eta ]
# ---------------------------------------------------------------------------

def fermi_dirac(j, eta):
    f = lambda x: (x ** j) / (1.0 + np.exp(np.clip(x - eta, -700, 700)))
    val, _ = quad(f, 0, np.inf, limit=200)
    return val

ETA_GRID = np.linspace(-15, 20, 400)
F0 = np.array([fermi_dirac(0.0, e) for e in ETA_GRID])
F1 = np.array([fermi_dirac(1.0, e) for e in ETA_GRID])
FHALF = np.array([fermi_dirac(0.5, e) for e in ETA_GRID])
S_GRID = (KB / E) * (2.0 * F1 / F0 - ETA_GRID)  # V/K, monotonically decreasing in eta

# n_reduced = F_{1/2}(eta) is monotone increasing -> invert via interpolation
eta_of_lnFhalf = interp1d(np.log(FHALF), ETA_GRID, bounds_error=False,
                           fill_value=(ETA_GRID[0], ETA_GRID[-1]))
S_of_eta = interp1d(ETA_GRID, S_GRID, bounds_error=False,
                     fill_value=(S_GRID[0], S_GRID[-1]))

def spb_seebeck(n_m3, T, mstar_ratio):
    """n in m^-3, T in K, mstar_ratio = m*/m_e. Returns S in V/K (signed like data,
    i.e. this returns the magnitude for n-type convention; sign handled by caller)."""
    mstar = mstar_ratio * ME
    Nc = 4.0 * np.pi * (2.0 * mstar * KB * T / H ** 2) ** 1.5
    n_reduced = n_m3 / Nc
    n_reduced = np.clip(n_reduced, 1e-6, None)
    eta = eta_of_lnFhalf(np.log(n_reduced))
    return S_of_eta(eta)


def fit_mstar(T, n_m3, S_abs):
    """Fit a single m*/m_e for a group of (T, n, |S|) points, least squares."""
    def loss(mstar_ratio):
        pred = spb_seebeck(n_m3, T, mstar_ratio)
        return np.sum((pred - S_abs) ** 2)
    res = minimize_scalar(loss, bounds=(0.02, 25.0), method="bounded")
    return res.x


# ---------------------------------------------------------------------------
# 2. Load samples + curves, parse sample_info, build matched (T, S, n) triples
# ---------------------------------------------------------------------------

def parse_info(info):
    if not isinstance(info, str) or not info.strip():
        return {}
    try:
        return json.loads(info)
    except Exception:
        return {}


def parse_arr(s):
    try:
        return np.array(ast.literal_eval(s), dtype=float)
    except Exception:
        return None


print("Loading samples table...")
samples = pd.read_csv(f"{DATA_DIR}/raw_starrydata2_samples.csv.gz")
samples["info"] = samples["sample_info"].apply(parse_info)
samples["family_tag"] = samples["info"].apply(lambda d: (d.get("MaterialFamily") or {}).get("category") or None)
samples["fab_cat"] = samples["info"].apply(lambda d: (d.get("FabricationProcess") or {}).get("category") or None)
samples["fab_comment"] = samples["info"].apply(lambda d: (d.get("FabricationProcess") or {}).get("comment") or None)
samples["density"] = samples["info"].apply(lambda d: (d.get("RelativeDensity") or {}).get("extracted") or None)
samples["grainsize"] = samples["info"].apply(lambda d: (d.get("GrainSize") or {}).get("extracted") or None)
samples = samples.drop(columns=["info"])

print("Loading curves table (this is the big one)...")
curves = pd.read_csv(f"{DATA_DIR}/raw_starrydata2_curves.csv.gz",
                      usecols=["sample_id", "prop_x", "prop_y", "x", "y", "composition"])

seebeck_curves = curves[(curves.prop_x == "Temperature") & (curves.prop_y == "Seebeck coefficient")]
carrier_curves = curves[(curves.prop_x == "Temperature") & (curves.prop_y == "Carrier concentration")]
hall_curves = curves[(curves.prop_x == "Temperature") & (curves.prop_y == "Hall coefficient")]

print(f"Seebeck(T) curves: {len(seebeck_curves)}   Carrier-conc(T) curves: {len(carrier_curves)}   "
      f"Hall(T) curves (as n proxy, n=1/(|R_H| e)): {len(hall_curves)}")

# group carrier curves by sample_id for lookup (a sample can have >1 curve, e.g. multiple figures).
# Hall coefficient(T) curves are converted to a carrier-concentration proxy via the
# single-carrier relation n = 1/(|R_H| * e) (units already m^3/C in this dataset -> n in m^-3)
# and pooled in as an additional source, since direct Carrier-concentration(T) curves alone
# are too sparse to build a usable Tier 1 pool (see the real-yield note below).
carrier_by_sample = {}


def add_points(sid, tx, ny):
    if sid not in carrier_by_sample:
        carrier_by_sample[sid] = ([], [])
    carrier_by_sample[sid][0].append(tx)
    carrier_by_sample[sid][1].append(ny)


for _, row in carrier_curves.iterrows():
    tx, ny = parse_arr(row["x"]), parse_arr(row["y"])
    if tx is None or ny is None or len(tx) < 2:
        continue
    add_points(row["sample_id"], tx, ny)

for _, row in hall_curves.iterrows():
    tx, rh = parse_arr(row["x"]), parse_arr(row["y"])
    if tx is None or rh is None or len(tx) < 2:
        continue
    rh = np.abs(rh)
    rh = np.where(rh > 0, rh, np.nan)
    n_from_hall = 1.0 / (rh * E)  # m^-3
    add_points(row["sample_id"], tx, n_from_hall)

carrier_by_sample = {
    sid: (np.concatenate(ts), np.concatenate(ns))
    for sid, (ts, ns) in carrier_by_sample.items()
}

print(f"Samples with a usable carrier-concentration(T) source (direct + Hall-derived): {len(carrier_by_sample)}")

# ---------------------------------------------------------------------------
# Composition-based family classifier — the MaterialFamily metadata tag is only
# filled in for ~39% of samples (a real, direct instance of the metadata-sparsity
# problem this thesis's literature review documents from the sysTEm paper), so
# family assignment falls back to composition-string heuristics, extending the
# half-Heusler/skutterudite regexes already written for this thesis.
# ---------------------------------------------------------------------------
import re

FAMILY_RULES = [
    ("Half-Heusler", re.compile(r"(Ti|Zr|Hf|V|Nb|Ta|Sc|Y|La|Ce)\w*.*(Ni|Co|Fe|Pd|Pt|Rh)\w*.*(Sn|Sb|Bi|As|Ge)")),
    ("Skutterudite", re.compile(r"(Co|Rh|Ir)\d*\.?\d*Sb\d*\.?\d{2,}")),
    ("Bi2Te3-based", re.compile(r"Bi\d*\.?\d*.*Te\d")),
    ("Sb2Te3/Ge-Sb-Te", re.compile(r"Sb\d*\.?\d*Te\d") ),
    ("PbTe-based", re.compile(r"Pb\d*\.?\d*Te\d")),
    ("SnSe-based", re.compile(r"Sn\d*\.?\d*Se\d")),
    ("SnS-based", re.compile(r"Sn\d*\.?\d*S\d(?!e)")),
    ("SnTe-based", re.compile(r"Sn\d*\.?\d*Te\d")),
    ("GeTe/GeSe", re.compile(r"Ge\d*\.?\d*(Te|Se)\d")),
    ("SiGe alloy", re.compile(r"^(Si\d*\.?\d*Ge\d*\.?\d*|Ge\d*\.?\d*Si\d*\.?\d*)$")),
    ("Silicide (CrSi2/FeSi2 etc.)", re.compile(r"(Cr|Fe|Mn|Mg)\d*\.?\d*Si\d")),
    ("Mg2Si/Sn", re.compile(r"Mg2(Si|Sn)")),
    ("Zintl (Ca/Sr/Eu/Yb-Zn/Al-Sb)", re.compile(r"(Ca|Sr|Eu|Yb)\d*\.?\d*.*(Zn|Al|Cd)\d*\.?\d*Sb")),
    ("Cu-chalcogenide (Cu2Se/CuInTe2 etc.)", re.compile(r"Cu\d*\.?\d*(Se|S|Te|In)")),
    ("In-Te/In-Se chalcogenide", re.compile(r"^In\d*\.?\d*(Te|Se)")),
    ("Ag-chalcogenide", re.compile(r"^Ag\d*\.?\d*(Ga|S|Se|Te)")),
    ("Oxide/Cobaltite", re.compile(r"(Co|Zn)\w*O\d|^Zn?O$|CrO|Pb\d*\.?\d*CrO")),
]


def classify_family(tag, composition):
    if isinstance(tag, str) and tag.strip():
        return tag
    comp = str(composition)
    for name, pat in FAMILY_RULES:
        if pat.search(comp):
            return name
    return None


samples["family_assigned"] = samples.apply(
    lambda r: classify_family(r["family_tag"], r["composition"]), axis=1)


def extract_matched_points(seebeck_row_group, min_T=150, max_T=1200):
    """For one sample_id's Seebeck(T) curve(s), interpolate the matching n(T)
    curve onto the same T grid (only within the n-curve's own T range)."""
    sid = seebeck_row_group.name
    if sid not in carrier_by_sample:
        return None
    Tn, nn = carrier_by_sample[sid]
    order = np.argsort(Tn)
    Tn, nn = Tn[order], nn[order]
    # de-duplicate T for interp1d
    Tn_u, idx = np.unique(Tn, return_index=True)
    nn_u = nn[idx]
    if len(Tn_u) < 2:
        return None
    n_interp = interp1d(Tn_u, nn_u, bounds_error=False, fill_value=np.nan)

    rows = []
    for _, row in seebeck_row_group.iterrows():
        Ts, Ss = parse_arr(row["x"]), parse_arr(row["y"])
        if Ts is None or Ss is None:
            continue
        for T, S in zip(Ts, Ss):
            if not (min_T <= T <= max_T):
                continue
            if T < Tn_u.min() or T > Tn_u.max():
                continue
            n_val = float(n_interp(T))
            if not np.isfinite(n_val) or n_val <= 0:
                continue
            rows.append((sid, T, S, n_val))
    return rows


print("Matching Seebeck and carrier-concentration curves per sample...")
all_rows = []
for sid, grp in seebeck_curves.groupby("sample_id"):
    grp.name = sid
    r = extract_matched_points(grp)
    if r:
        all_rows.extend(r)

matched = pd.DataFrame(all_rows, columns=["sample_id", "T", "S_measured", "n_measured"])
print(f"Matched (T, S, n) points before quality filtering: {len(matched)}")

# unit sanity: |S| should be < ~0.002 V/K (2000 uV/K) for real TE materials;
# n should be within a broad physically sane window for doped semiconductors
matched = matched[(matched["S_measured"].abs() < 0.002) &
                   (matched["n_measured"] > 1e18) & (matched["n_measured"] < 1e27)]
print(f"Matched points after CORRECTED physical-range filtering (n in [1e18,1e27] m^-3, doped-TE window): {len(matched)}")

matched = matched.merge(
    samples[["sample_id", "composition", "family_tag", "family_assigned", "fab_cat", "fab_comment", "density", "grainsize"]],
    on="sample_id", how="left")

# ---------------------------------------------------------------------------
# TIER 1 — pooled multi-family set
# ---------------------------------------------------------------------------
tier1 = matched.dropna(subset=["family_assigned"]).copy()
tier1 = tier1.rename(columns={"family_assigned": "family"})
family_counts = tier1.groupby("family")["sample_id"].nunique().sort_values(ascending=False)
KEEP_FAMILIES = family_counts[family_counts >= 3].index.tolist()
tier1 = tier1[tier1["family"].isin(KEEP_FAMILIES)].copy()

print(f"\nTIER 1: {tier1['sample_id'].nunique()} samples, {tier1['family'].nunique()} families "
      f"(family kept if >=3 distinct samples), {len(tier1)} (T,S,n) points")

# log-n descriptor (carrier concentration in cm^-3 for readability)
tier1["n_cm3"] = tier1["n_measured"] / 1e6
tier1["log10_n"] = np.log10(tier1["n_cm3"])

# Fit m*/m_e per family, STRATIFIED BY CARRIER TYPE (sign of measured Seebeck).
# Pooling n-type and p-type carriers under one shared m* is not physically
# defensible (conduction- vs valence-band effective mass genuinely differ), and
# empirically it is exactly what was driving the optimizer to saturate at the
# search bounds instead of finding an interior fit. Fall back to a pooled fit
# only when a sign-subgroup is too small (<3 points) to fit on its own.
tier1_out = []
mstar_by_group = {}
for (fam, sgn), grp in tier1.groupby(["family", np.sign(tier1["S_measured"])]):
    T = grp["T"].values
    n_m3 = grp["n_measured"].values
    S_abs = grp["S_measured"].abs().values
    if len(grp) >= 3:
        mstar = fit_mstar(T, n_m3, S_abs)
    else:
        # too few points of this carrier type in this family: fall back to the
        # family-level pooled fit (still better than an arbitrary default of 1.0)
        fam_grp = tier1[tier1["family"] == fam]
        mstar = fit_mstar(fam_grp["T"].values, fam_grp["n_measured"].values,
                           fam_grp["S_measured"].abs().values) if len(fam_grp) >= 3 else 1.0
    mstar_by_group[(fam, sgn)] = mstar
    pred = spb_seebeck(n_m3, T, mstar)
    signed_pred = np.sign(grp["S_measured"].values) * pred
    g = grp.copy()
    g["carrier_type"] = np.where(sgn > 0, "p-type", "n-type")
    g["mstar_over_me_family_fit"] = mstar
    g["S_baseline_V_per_K"] = signed_pred
    g["S_residual_V_per_K"] = g["S_measured"] - g["S_baseline_V_per_K"]
    tier1_out.append(g)
tier1_final = pd.concat(tier1_out, ignore_index=True)

resid_uVK = tier1_final["S_residual_V_per_K"] * 1e6
rmse = np.sqrt(np.mean(resid_uVK ** 2))
mae = np.mean(np.abs(resid_uVK))
print(f"TIER 1 SPB baseline fit (per-family m*): RMSE={rmse:.1f} uV/K, MAE={mae:.1f} uV/K "
      f"(measured |S| median = {tier1_final['S_measured'].abs().median()*1e6:.1f} uV/K)")

# family-wise leave-one-family-out fold assignment (for use downstream)
families_sorted = sorted(tier1_final["family"].unique())
tier1_final["loFo_fold"] = tier1_final["family"].apply(lambda f: families_sorted.index(f))

tier1_final.to_csv(f"{OUT_DIR}/tier1_pooled_baseline_residuals.csv", index=False)

# ---------------------------------------------------------------------------
# TIER 2 — half-Heusler
# ---------------------------------------------------------------------------
hh_all = pd.read_csv(f"{UPLOAD_DIR}/tier2_input_halfheusler_samples.csv")
hh_ids = set(hh_all["sample_id"])
tier2 = matched[matched["sample_id"].isin(hh_ids)].copy()
tier2["n_cm3"] = tier2["n_measured"] / 1e6
tier2["log10_n"] = np.log10(tier2["n_cm3"])

n_samples_t2 = tier2["sample_id"].nunique()
n_with_micro = tier2.dropna(subset=["density"])["sample_id"].nunique() + \
               tier2.dropna(subset=["grainsize"])["sample_id"].nunique()
print(f"\nTIER 2 (half-Heusler): {n_samples_t2} samples with matched (T,S,n) points, "
      f"{len(tier2)} points total")

# Same carrier-type stratification as Tier 1 (see comment above): half-Heusler
# pools genuine n-type and p-type compositions, so fit m* separately per sign.
tier2_out = []
for sgn, grp in tier2.groupby(np.sign(tier2["S_measured"])):
    T = grp["T"].values
    n_m3 = grp["n_measured"].values
    S_abs = grp["S_measured"].abs().values
    mstar = fit_mstar(T, n_m3, S_abs) if len(grp) >= 3 else 1.0
    pred = spb_seebeck(n_m3, T, mstar)
    g = grp.copy()
    g["carrier_type"] = np.where(sgn > 0, "p-type", "n-type")
    g["mstar_over_me_fit"] = mstar
    g["S_baseline_V_per_K"] = np.sign(g["S_measured"].values) * pred
    g["S_residual_V_per_K"] = g["S_measured"] - g["S_baseline_V_per_K"]
    tier2_out.append(g)
tier2 = pd.concat(tier2_out, ignore_index=True)

resid2_uVK = tier2["S_residual_V_per_K"] * 1e6
print(f"TIER 2 SPB baseline fit (per-carrier-type m*): "
      f"RMSE={np.sqrt(np.mean(resid2_uVK**2)):.1f} uV/K, MAE={np.mean(np.abs(resid2_uVK)):.1f} uV/K")
for ct, g in tier2.groupby("carrier_type"):
    print(f"    {ct}: n={len(g)} pts, m*={g['mstar_over_me_fit'].iloc[0]:.3f} m_e, "
          f"RMSE={np.sqrt(np.mean((g['S_residual_V_per_K']*1e6)**2)):.1f} uV/K")

# leave-one-out fold assignment at the SAMPLE level (not point level) so a sample's
# points never span train/test
sample_ids_t2 = sorted(tier2["sample_id"].unique())
tier2["loo_fold"] = tier2["sample_id"].apply(lambda s: sample_ids_t2.index(s))

tier2.to_csv(f"{OUT_DIR}/tier2_halfheusler_baseline_residuals.csv", index=False)

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print("\n=== FAMILY BREAKDOWN (Tier 1) ===")
print(tier1_final.groupby("family")["sample_id"].nunique().sort_values(ascending=False).to_string())

print("\nWrote:")
print(f"  {OUT_DIR}/tier1_pooled_baseline_residuals.csv  ({len(tier1_final)} rows)")
print(f"  {OUT_DIR}/tier2_halfheusler_baseline_residuals.csv  ({len(tier2)} rows)")


Loading samples table...
Loading curves table (this is the big one)...
Seebeck(T) curves: 36112   Carrier-conc(T) curves: 1921   Hall(T) curves (as n proxy, n=1/(|R_H| e)): 587
Samples with a usable carrier-concentration(T) source (direct + Hall-derived): 2110
Matching Seebeck and carrier-concentration curves per sample...
Matched (T, S, n) points before quality filtering: 18968
Matched points after CORRECTED physical-range filtering (n in [1e18,1e27] m^-3, doped-TE window): 16052

TIER 1: 967 samples, 37 families (family kept if >=3 distinct samples), 12358 (T,S,n) points
TIER 1 SPB baseline fit (per-family m*): RMSE=138.8 uV/K, MAE=71.7 uV/K (measured |S| median = 158.4 uV/K)

TIER 2 (half-Heusler): 55 samples with matched (T,S,n) points, 880 points total
TIER 2 SPB baseline fit (per-carrier-type m*): RMSE=257.6 uV/K, MAE=89.6 uV/K
    n-type: n=666 pts, m*=1.886 m_e, RMSE=290.7 uV/K
    p-type: n=214 pts, m*=0.515 m_e, RMSE=99.5 uV/K

=== FAMILY BREAKDOWN (Tier 1) ===
family
Bi2Te3-

## Step 5: independent cross-check against DFT (Ricci ab initio database)

The corrected Tier 2 fit gives `m*_p = 0.515 m_e` and `m*_n = 1.886 m_e` for
half-Heusler. This step checks that against effective masses derived
independently from DFT-computed (Seebeck, T, carrier-concentration) triples
in the Ricci database — same SPB physics, run in the opposite direction
(solve for m* given S, instead of solve for S given m*), with **no fitting
to the experimental data at all**.

The DFT database reports several evaluation points per compound at fixed
carrier-concentration levels (1e16, 1e18, 1e19, 1e20 cm⁻³). Using every
point gives unstable, near-zero medians because the 1e16 cm⁻³ points are
close to intrinsic (non-degenerate) doping, where the SPB inversion is
poorly conditioned. Restricting to `c = 1e20 cm⁻³` — the degenerate,
heavily-doped level that is actually comparable to real thermoelectric
operating conditions — gives a stable answer. Both are shown below so the
instability itself is on the record, not hidden.

In [ ]:
import openpyxl
import pandas as pd, numpy as np
from scipy.integrate import quad
from scipy.interpolate import interp1d

KB = 1.380649e-23; E = 1.602176634e-19; H = 6.62607015e-34; ME = 9.1093837015e-31

def fermi_dirac(j, eta):
    f = lambda x: (x ** j) / (1.0 + np.exp(np.clip(x - eta, -700, 700)))
    val, _ = quad(f, 0, np.inf, limit=200)
    return val

ETA_GRID = np.linspace(-15, 20, 400)
F0 = np.array([fermi_dirac(0.0, e) for e in ETA_GRID])
F1 = np.array([fermi_dirac(1.0, e) for e in ETA_GRID])
S_GRID = (KB / E) * (2.0 * F1 / F0 - ETA_GRID)
S_of_eta = interp1d(ETA_GRID, S_GRID, bounds_error=False, fill_value=(S_GRID[0], S_GRID[-1]))
eta_of_S = interp1d(S_GRID[::-1], ETA_GRID[::-1], bounds_error=False, fill_value=(ETA_GRID[-1], ETA_GRID[0]))

def mstar_from_S_T_c(S_uVK, T, c_cm3):
    """Invert the SPB relation: given a (Seebeck, temperature, carrier-concentration)
    triple, solve for the effective mass m*/m_e that the SPB model would need to
    reproduce that Seebeck value at that T and n. This is the SAME physics as the
    experimental fit (build_tiers_cleaned.py), just run in the opposite direction:
    there we fix m* and predict S; here we fix S (from DFT) and solve for m*."""
    S_VK = abs(S_uVK) * 1e-6
    n_m3 = c_cm3 * 1e6
    eta = eta_of_S(S_VK)
    F0_val = fermi_dirac(0.0, eta)
    Nc_needed = n_m3 / F0_val
    mstar = (Nc_needed / (4*np.pi))**(2.0/3.0) * H**2 / (2*KB*T) / ME
    return mstar

wb = openpyxl.load_workbook('/root/.claude/uploads/5c312dbd-3c58-586b-b622-94f720818679/982343af-merged-output.xlsx', read_only=True, data_only=True)

for sheet, label in [('ricci_dft_clean_halfheusler_1', 'half-Heusler'), ('ricci_dft_clean_skutterudite_1', 'skutterudite')]:
    ws = wb[sheet]
    rows = list(ws.iter_rows(values_only=True))
    cols = rows[0]
    df = pd.DataFrame(rows[1:], columns=cols)
    print(f'=== {label} (n={len(df)} DFT entries) ===')
    for prefix, ctype in [('Sᵉ.p', 'p-type (hole)'), ('Sᵉ.n', 'n-type (electron)')]:
        v, T, c = f'{prefix}.v [µV/K]', f'{prefix}.T [K]', f'{prefix}.c [cm⁻³]'
        sub = df[[v, T, c]].apply(pd.to_numeric, errors='coerce').dropna()

        # --- ALL evaluation points (includes near-intrinsic 1e16 cm^-3 rows) ---
        mstars_all = []
        for _, row in sub.iterrows():
            try:
                m = mstar_from_S_T_c(row[v], row[T], row[c])
                if np.isfinite(m) and 0 < m < 1000:
                    mstars_all.append(m)
            except Exception:
                pass
        mstars_all = np.array(mstars_all)

        # --- restricted to c = 1e20 cm^-3 only (the degenerate, experimentally-comparable doping level) ---
        sub_1e20 = sub[np.isclose(sub[c], 1e20)]
        mstars_1e20 = []
        for _, row in sub_1e20.iterrows():
            try:
                m = mstar_from_S_T_c(row[v], row[T], row[c])
                if np.isfinite(m) and 0 < m < 1000:
                    mstars_1e20.append(m)
            except Exception:
                pass
        mstars_1e20 = np.array(mstars_1e20)

        print(f'  {ctype}:')
        print(f'    all c-levels   : n={len(mstars_all)}, median m*={np.median(mstars_all):.3f}, '
              f'p25={np.percentile(mstars_all,25):.3f}, p75={np.percentile(mstars_all,75):.3f}')
        if len(mstars_1e20) > 0:
            print(f'    c=1e20 cm^-3 only: n={len(mstars_1e20)}, median m*={np.median(mstars_1e20):.3f}, '
                  f'p25={np.percentile(mstars_1e20,25):.3f}, p75={np.percentile(mstars_1e20,75):.3f}')
        else:
            print(f'    c=1e20 cm^-3 only: no rows at exactly this level')
    print()


=== half-Heusler (n=153 DFT entries) ===
  p-type (hole):
    all c-levels   : n=153, median m*=0.153, p25=0.001, p75=0.944
    c=1e20 cm^-3 only: n=58, median m*=0.226, p25=0.114, p75=0.904
  n-type (electron):
    all c-levels   : n=153, median m*=0.428, p25=0.002, p75=0.987
    c=1e20 cm^-3 only: n=58, median m*=0.944, p25=0.347, p75=0.944

=== skutterudite (n=22 DFT entries) ===
  p-type (hole):
    all c-levels   : n=22, median m*=0.093, p25=0.001, p75=0.444
    c=1e20 cm^-3 only: n=5, median m*=0.361, p25=0.325, p75=0.472
  n-type (electron):
    all c-levels   : n=22, median m*=0.003, p25=0.000, p75=1.257
    c=1e20 cm^-3 only: n=4, median m*=1.213, p25=0.975, p75=1.534



**Reading this result:** at c=1e20 cm⁻³, DFT gives half-Heusler
`m*_p ≈ 0.226` (IQR 0.114-0.904, n=58) and `m*_n ≈ 0.944` (IQR 0.347-0.944,
n=58). The experimental fit's p-type value (0.515) sits inside the DFT
interquartile range. The experimental n-type value (1.886) sits clearly
above the DFT median (0.944) and above the DFT IQR's upper bound — a real,
reportable discrepancy, not just noise, and worth a sentence in the thesis
(plausible causes: the experimental fit pools many different n-type
half-Heusler compositions under one shared mass, while DFT values vary a
lot compound-to-compound; or a real scattering-mechanism mismatch the plain
SPB model doesn't capture). Skutterudite has far fewer DFT points (n=4-5 at
c=1e20) so those numbers are indicative only, not a strong test.

## Step 6: does the residual relate to fabrication process?

One residual value per **sample** (mean |residual| across that sample's
points, to avoid treating correlated points from the same sample as
independent observations), family-normalized by taking the log and
z-scoring within each family, then a Kruskal-Wallis test across process
categories with at least 5 samples.

**Note on this number vs. an earlier, informally-reported one:** an earlier
pass in this project reported H=18.73, p=0.028 for this same question, but
the exact script that produced that number was not saved to disk, so it
could not be re-run to confirm it. The script below is a fresh, from-scratch
reconstruction of the same test, run and captured just now. It gives a
noticeably different result (below) — most likely because of a different
choice of aggregation (per-sample vs. per-point) or z-scoring detail that
wasn't recorded. Per the instruction not to state unverified numbers: treat
the result in this notebook, not the earlier 0.028 figure, as the current,
actually-reproduced answer, until the two are reconciled.

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

df = pd.read_csv('/tmp/verify_thesis/tier1_pooled_baseline_residuals.csv')

# One residual value per SAMPLE (not per point), since Kruskal-Wallis needs
# independent observations and a sample contributes many correlated (T,S,n) points.
per_sample = (df.groupby('sample_id')
                .agg(family=('family', 'first'),
                     fab_cat=('fab_cat', 'first'),
                     resid=('S_residual_V_per_K', lambda x: np.mean(np.abs(x))))
              .reset_index())
per_sample = per_sample.dropna(subset=['fab_cat'])

print(f"Samples with a fabrication-process label: {len(per_sample)} / {df['sample_id'].nunique()} total Tier-1 samples")

# Family-normalize: log the residual, then z-score WITHIN each family, so a
# "big" residual in one family isn't compared raw against a different family's scale.
per_sample['log_resid'] = np.log(per_sample['resid'] + 1e-12)
per_sample['z'] = per_sample.groupby('family')['log_resid'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() > 0 and len(x) > 1 else 0.0
)

counts = per_sample['fab_cat'].value_counts()
keep_cats = counts[counts >= 5].index.tolist()
sub = per_sample[per_sample['fab_cat'].isin(keep_cats)]

print(f"Process categories with n>=5: {keep_cats}")
print(f"Total samples in test: {len(sub)}")
print()
print("Per-category mean z-scored log-residual (negative = closer to ideal SPB behavior):")
print(sub.groupby('fab_cat')['z'].agg(['count', 'mean']).sort_values('mean').to_string())
print()

groups = [g['z'].values for _, g in sub.groupby('fab_cat')]
H, p = stats.kruskal(*groups)
print(f"Kruskal-Wallis: H={H:.2f}, p={p:.4f}, df={len(groups)-1}")


Samples with a fabrication-process label: 377 / 967 total Tier-1 samples
Process categories with n>=5: ['PressureSintering', 'Sintering', 'HighPressureSynthesis', 'MeltGrowth', 'MeltQuench', 'Other', 'FilmGrowth', 'VaporGrowth', 'Melting', 'FluxGrowth']
Total samples in test: 371

Per-category mean z-scored log-residual (negative = closer to ideal SPB behavior):
                       count      mean
fab_cat                               
MeltQuench                24 -0.220291
HighPressureSynthesis     34 -0.177089
PressureSintering        179 -0.115023
Sintering                 37  0.017581
FluxGrowth                 8  0.152174
Other                     17  0.195408
VaporGrowth               11  0.198507
FilmGrowth                17  0.212367
MeltGrowth                34  0.409769
Melting                   10  0.477119

Kruskal-Wallis: H=16.61, p=0.0551, df=9


**Reading this result:** H=16.61, p=0.055 across 10 process
categories (371 samples) — a suggestive but not-quite-significant-at-0.05
association between fabrication process and residual size. The direction is
physically sensible: pressure-based and high-pressure routes
(`PressureSintering`, `HighPressureSynthesis`, `MeltQuench`) skew toward
*smaller* residuals (closer to ideal SPB behavior), while melt-based routes
(`MeltGrowth`, `Melting`) skew toward *larger* residuals. That pattern
matches the earlier informal finding's direction even though the exact
H/p values differ, which is a reason to think the underlying signal is real,
but the test itself needs to be firmed up (larger per-category samples,
and/or controlling for temperature and family more explicitly) before
treating p=0.05ish as a result to report.

## Where this leaves the project

Steps 1-5 are solid: the pipeline is reproducible end-to-end from raw data,
both known bugs are fixed, and the physics baseline has now been checked
against an independent (DFT) source rather than only self-consistency.

Step 6 is the one piece that is not yet settled — the association between
residual and fabrication process is real in direction but currently sits
right at the edge of conventional significance, and the exact figure needs
to be pinned down with one canonical script (this notebook's version, going
forward) rather than an unreproducible earlier number. This is also exactly
the point where the next decision (symbolic regression restricted to
physically-sensible forms, benchmarked against XGBoost) plugs in — modeling
the residual against family/process/carrier-type properly, instead of one
single non-parametric test, is the natural way to firm this up.